In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# Load data from data_processed_original_only/ (split first, no augmentation)
# ============================================================================

data_path = Path('data_processed_original_only')

# Verify directory exists
assert data_path.exists(), f"Data directory not found: {data_path}"

# Load training set (ORIGINAL images only)
X_train = np.load(data_path / 'train' / 'images.npy')
y_train = np.load(data_path / 'train' / 'labels.npy')

# Load validation set (ORIGINAL images)
X_val = np.load(data_path / 'val' / 'images.npy')
y_val = np.load(data_path / 'val' / 'labels.npy')

# Load test set (ORIGINAL images)
X_test = np.load(data_path / 'test' / 'images.npy')
y_test = np.load(data_path / 'test' / 'labels.npy')

# Load metadata
with open(data_path / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print("✅ Data Loaded Successfully!")
print(f"\nDataset Shape:")
print(f"  Train: {X_train.shape} | Labels: {np.bincount(y_train)}")
print(f"  Val:   {X_val.shape} | Labels: {np.bincount(y_val)}")
print(f"  Test:  {X_test.shape} | Labels: {np.bincount(y_test)}")
print(f"\n📌 Train set uses ORIGINAL images only (no augmentation)")
print(f"📌 Val & Test sets are also ORIGINAL")

In [ ]:
# ImageNet normalization parameters
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def normalize_imagenet(images):
    """
    Apply ImageNet normalization
    images: (N, 224, 224, 3) with values in [0, 1]
    returns: (N, 224, 224, 3) normalized
    """
    normalized = np.zeros_like(images)
    for i in range(3):  # RGB channels
        normalized[..., i] = (images[..., i] - IMAGENET_MEAN[i]) / IMAGENET_STD[i]
    return normalized

# Apply normalization
print("Applying ImageNet normalization...")
X_train_norm = normalize_imagenet(X_train)
X_val_norm = normalize_imagenet(X_val)
X_test_norm = normalize_imagenet(X_test)

# Convert to PyTorch tensors and reshape (N, 3, 224, 224) for PyTorch
# PyTorch expects: (Batch, Channels, Height, Width)
X_train_tensor = torch.from_numpy(X_train_norm.transpose(0, 3, 1, 2)).float()
y_train_tensor = torch.from_numpy(y_train).long()

X_val_tensor = torch.from_numpy(X_val_norm.transpose(0, 3, 1, 2)).float()
y_val_tensor = torch.from_numpy(y_val).long()

X_test_tensor = torch.from_numpy(X_test_norm.transpose(0, 3, 1, 2)).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Create PyTorch datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("✅ Normalization complete and tensors created")
print(f"   Train tensor: {X_train_tensor.shape} | dtype: {X_train_tensor.dtype}")
print(f"   Val tensor: {X_val_tensor.shape} | dtype: {X_val_tensor.dtype}")
print(f"   Test tensor: {X_test_tensor.shape} | dtype: {X_test_tensor.dtype}")

In [ ]:
# Create data loaders
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("✅ Data loaders created")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

In [ ]:
# Define simple CNN model (without ResNet)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=2).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ SimpleCNN model created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# Training configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    threshold=1e-4
)

NUM_EPOCHS = 50
PATIENCE = 10  # Early stopping

print("✅ Training configuration ready")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Early stopping patience: {PATIENCE}")

In [ ]:
from tqdm import tqdm

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_val_loss = float('inf')
patience_counter = 0

print("🚀 Starting CNN training...\n")
start_time = datetime.now()

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [TRAIN]", leave=False)
    for images, labels in train_pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [VAL]", leave=False)
        for images, labels in val_pbar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        patience_counter = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss
        }, 'cnn_only_best.pth')
        print(f"  ✅ Best model saved: {val_acc:.2f}%")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print(f"  ⛔ Early stopping at epoch {epoch+1}")
        break

elapsed_time = datetime.now() - start_time
print(f"\n✅ Training complete! Time: {elapsed_time}")

In [ ]:
# Evaluate on held-out test set
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (preds == labels).sum().item()

test_acc = 100 * test_correct / test_total
print(f"✅ Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Save final model and metadata
torch.save(model.state_dict(), 'cnn_only_final.pth')

training_metadata = {
    'model': 'SimpleCNN',
    'dataset': 'data_processed_original_only',
    'cv_enabled': False,
    'train_images': int(X_train.shape[0]),
    'val_images': int(X_val.shape[0]),
    'test_images': int(X_test.shape[0]),
    'best_val_accuracy': float(best_val_acc),
    'best_val_loss': float(best_val_loss),
    'test_accuracy': float(test_acc),
    'total_epochs': len(history['train_loss']),
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.001,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'training_date': datetime.now().isoformat(),
    'device': str(device),
    'training_time': str(elapsed_time),
    'history': history
}

with open('cnn_only_metadata.json', 'w') as f:
    json.dump(training_metadata, f, indent=2)

print('✅ Saved: cnn_only_final.pth')
print('✅ Saved: cnn_only_best.pth')
print('✅ Saved: cnn_only_metadata.json')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('CNN Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
axes[1].set_title('CNN Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_only_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Saved: cnn_only_training_history.png')